# Phonetic Guided Speech Generation
### Explicit phoneme alignment (AML) + learned acoustic generation (DL) on LJSpeech

**Question.** If we know exactly *when* every phoneme happens, how much good speech do we get for free,
and how much needs a learned acoustic model?

**Pipeline.** Montreal Forced Aligner → LightGBM duration predictor + LightGBM pause classifier (AML)
→ timing plan → rendered either by an averaged-mel baseline (AML only) or by a FastSpeech2 decoder
conditioned on our durations, then fine-tuned (DL) → HiFi-GAN vocoder.

**How to run.** Google Colab, **T4 GPU** (Runtime → Change runtime type). Run the first cell,
**restart the session once**, then run everything top to bottom (~1.5 h on a T4).
Everything is downloaded from official sources or from this repository's GitHub Release.

| Section | What |
|---|---|
| 0 | Setup |
| 1 | Data: LJSpeech + cached mel spectrograms |
| 2 | Forced alignments (MFA) |
| 3 | Duration table, splits, leak-free evaluation set |
| 4 | AML model 1: duration predictor |
| 5 | AML model 2: pause classifier |
| 6 | Synthesis systems (baseline, hybrid, FastSpeech2) |
| 7 | Fine-tuning the decoder |
| 8 | Rendering all systems |
| 9 | Evaluation: WER, CIs, duration MAE, UTMOS |
| 10 | Human listening test |
| 11 | Figures |
| 12 | Listen to samples |
| A | Appendix: running MFA yourself |

## 0. Setup

In [ ]:
!pip install -q speechbrain lightgbm textgrid openai-whisper jiwer soundfile pyyaml
print("Done. Now: Runtime -> Restart session (once), then run the cells below.")

In [ ]:
REPO_URL = "https://github.com/<your-username>/phonetic-guided-tts.git"
RELEASE  = REPO_URL[:-4] + "/releases/download/v1.0"
RAW      = REPO_URL.replace("github.com", "raw.githubusercontent.com")[:-4] + "/main"
USE_DRIVE = False       # True = keep large intermediate files on Google Drive across disconnects

import os, re, glob, json, time, random, shutil
import numpy as np, pandas as pd, torch, torchaudio, soundfile as sf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import Audio, display

cfg = {
    "paths": {"data_root": "/content/LJSpeech-1.1", "align_dir": "/content/LJ_aligned",
              "mel_dir": "/content/mels", "work_dir": "/content/work",
              "results_dir": "/content/outputs/results", "samples_dir": "/content/outputs/samples"},
    "audio": {"sr": 22050, "n_fft": 1024, "win_length": 1024, "hop_length": 256, "n_mels": 80,
              "f_min": 0.0, "f_max": 8000.0},
    "split": {"test_size": 0.1, "seed": 42},
    "speechbrain_split": {"seed": 1234, "ratio": [90, 10],
        "prepare_script_url": "https://raw.githubusercontent.com/speechbrain/speechbrain/develop/recipes/LJSpeech/ljspeech_prepare.py"},
    "duration_model": {"n_estimators": 600, "learning_rate": 0.05, "num_leaves": 63},
    "pause_model": {"min_pause_sec": 0.10, "top_words": 80, "n_estimators": 500, "learning_rate": 0.05,
                    "num_leaves": 63, "dev_fraction": 0.1, "seed": 0},
    "models": {"fastspeech2": "speechbrain/tts-fastspeech2-ljspeech", "hifigan": "speechbrain/tts-hifigan-ljspeech",
               "whisper": "small.en", "utmos_repo": "tarepan/SpeechMOS:v1.2.0", "utmos_model": "utmos22_strong"},
    "finetune": {"lr": 5e-5, "weight_decay": 1e-6, "epochs": 6, "batch_size": 16, "dev_size": 200,
                 "grad_clip": 1.0, "seed": 0},
    "eval": {"n_sentences": 100, "seed": 0, "n_bootstrap": 5000, "n_samples": 5},
}
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cfg["paths"]["work_dir"] = "/content/drive/MyDrive/phonetic-guided-tts-work"
for k in ("work_dir", "results_dir", "samples_dir"):
    os.makedirs(cfg["paths"][k], exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
# ---- helpers ----
import os
import random

import numpy as np


def work_path(cfg, *parts):
    """Path inside the (large, not committed) work directory; creates parent folders."""
    path = os.path.join(cfg["paths"]["work_dir"], *parts)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    return path


def results_path(cfg, name):
    """Path inside the (small, committed) results directory."""
    return os.path.join(cfg["paths"]["results_dir"], name)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)


def norm_audio(wav, peak=0.9):
    """Peak-normalise to avoid clipping when saving or playing audio."""
    return wav / (np.abs(wav).max() + 1e-9) * peak


def frame_ms(cfg):
    a = cfg["audio"]
    return 1000.0 * a["hop_length"] / a["sr"]


def device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"

## 1. Data: LJSpeech and cached mel spectrograms
Mel settings match the pretrained HiFi-GAN vocoder. Mels are cached as float16 so every experiment uses identical inputs.

In [ ]:
"""LJSpeech loading and log-mel extraction (settings match the pretrained HiFi-GAN)."""
import os
import tarfile
import urllib.request

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from tqdm.auto import tqdm

LJSPEECH_URL = "https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2"


def download_ljspeech(data_root):
    """Download and extract LJSpeech 1.1 if it is not already present."""
    if os.path.exists(os.path.join(data_root, "metadata.csv")):
        return
    parent = os.path.dirname(data_root.rstrip("/"))
    archive = os.path.join(parent, "LJSpeech-1.1.tar.bz2")
    if not os.path.exists(archive):
        print("Downloading LJSpeech (2.6 GB)...")
        urllib.request.urlretrieve(LJSPEECH_URL, archive)
    print("Extracting...")
    with tarfile.open(archive, "r:bz2") as tar:
        tar.extractall(parent)


def load_metadata(data_root):
    """id | raw text | normalised text. We always use the normalised column."""
    return pd.read_csv(os.path.join(data_root, "metadata.csv"), sep="|", header=None,
                       names=["id", "raw", "text"], quoting=3, keep_default_na=False)


class MelExtractor:
    """Natural-log mel spectrogram, clamped at 1e-5 - the format HiFi-GAN expects."""

    def __init__(self, audio_cfg, device="cpu"):
        a = audio_cfg
        self.sr, self.device = a["sr"], device
        self.tf = torchaudio.transforms.MelSpectrogram(
            sample_rate=a["sr"], n_fft=a["n_fft"], win_length=a["win_length"],
            hop_length=a["hop_length"], n_mels=a["n_mels"], f_min=a["f_min"], f_max=a["f_max"],
            power=1.0, normalized=False, norm="slaney", mel_scale="slaney").to(device)

    @torch.no_grad()
    def __call__(self, wav_path):
        wav, sr = sf.read(wav_path, dtype="float32")
        assert sr == self.sr, f"{wav_path}: expected {self.sr} Hz, got {sr}"
        mel = self.tf(torch.from_numpy(wav).to(self.device))
        return torch.log(torch.clamp(mel, min=1e-5))          # (n_mels, T)


def cache_mels(cfg, ids, device="cpu"):
    """Save every utterance's mel as float16 .npy so experiments are reproducible and fast."""
    mel_dir = cfg["paths"]["mel_dir"]
    os.makedirs(mel_dir, exist_ok=True)
    ext = MelExtractor(cfg["audio"], device)
    for u in tqdm(ids, desc="caching mels"):
        path = os.path.join(mel_dir, f"{u}.npy")
        if not os.path.exists(path):
            mel = ext(os.path.join(cfg["paths"]["data_root"], "wavs", f"{u}.wav"))
            np.save(path, mel.half().cpu().numpy())


def load_mel(cfg, u):
    """Cached mel as float32 (n_mels, T)."""
    return np.load(os.path.join(cfg["paths"]["mel_dir"], f"{u}.npy")).astype(np.float32)

In [ ]:
download_ljspeech(cfg["paths"]["data_root"])
meta = load_metadata(cfg["paths"]["data_root"])
print(f"{len(meta)} clips")
cache_mels(cfg, meta.id, device=device())
print("mels cached in", cfg["paths"]["mel_dir"])

## 2. Forced alignments (Montreal Forced Aligner)
MFA needs conda, which breaks PyTorch in the same Colab runtime, so alignment was run once separately
(**Appendix A**) and the TextGrids are attached to this repository's GitHub Release.

In [ ]:
!wget -q -nc {RELEASE}/LJ_aligned.tar -O /content/LJ_aligned.tar
!tar -xf /content/LJ_aligned.tar -C /content/
print(len(glob.glob("/content/LJ_aligned/*.TextGrid")), "TextGrids (13,100 expected)")

## 3. Duration table, splits and the leak-free evaluation set
* **Duration table:** one row per phoneme, with context features and its duration in mel frames.
* **Gap table:** one row per gap between two words: pause (silence ≥ 100 ms) or not, with text features.
* **Our split:** 90/10 by utterance.
* **Leak-free evaluation:** the pretrained FastSpeech2/HiFi-GAN were trained on SpeechBrain's own 90/10 split
  (seed 1234). We re-run SpeechBrain's split function to find the sentences they never trained on, and evaluate
  on 100 sentences that are in **our** test split **and** held out from **their** training.

In [ ]:
"""Turn MFA TextGrids into training tables.

* Duration table: one row per phoneme with context features and its duration in mel frames.
* Gap table: one row per gap between two words, labelled pause / no pause, with text features.
"""
import difflib
import glob
import os
import re

import pandas as pd
import textgrid
from tqdm.auto import tqdm


def to_frame(t, sr, hop):
    return int(round(t * sr / hop))


def strip_stress(mark):
    return re.sub(r"\d", "", mark)


def read_textgrid(align_dir, u):
    tg = textgrid.TextGrid.fromFile(os.path.join(align_dir, f"{u}.TextGrid"))
    return list(tg.getFirst("words")), list(tg.getFirst("phones"))


# ------------------------------------------------------------------ duration table
def utterance_duration_rows(u, words, phones, sr, hop):
    labels = [strip_stress(p.mark) or "sil" for p in phones]
    n, rows = len(phones), []
    for i, p in enumerate(phones):
        w = next((w for w in words if w.minTime <= p.minTime < w.maxTime), None)
        in_word = [q for q in phones if w and w.minTime <= q.minTime < w.maxTime and q.mark]
        rows.append(dict(
            utt=u, idx=i, n_phones=n, phone=labels[i],
            stress=int(p.mark[-1]) if p.mark[-1:].isdigit() else -1,
            prev=labels[i - 1] if i > 0 else "<s>",
            next=labels[i + 1] if i < n - 1 else "</s>",
            pos_in_word=in_word.index(p) if p in in_word else -1,
            word_len=len(in_word),
            before_pause=int(i < n - 1 and labels[i + 1] == "sil"),
            rel_pos=i / n,
            edge_sil=int(labels[i] == "sil" and i in (0, n - 1)),
            dur_sec=p.maxTime - p.minTime,
            frames=to_frame(p.maxTime, sr, hop) - to_frame(p.minTime, sr, hop)))
    return rows


def build_duration_table(align_dir, sr, hop):
    rows = []
    files = sorted(glob.glob(os.path.join(align_dir, "*.TextGrid")))
    for f in tqdm(files, desc="duration table"):
        u = os.path.basename(f)[: -len(".TextGrid")]
        words, phones = read_textgrid(align_dir, u)
        rows += utterance_duration_rows(u, words, phones, sr, hop)
    return pd.DataFrame(rows)


# ------------------------------------------------------------------ gap (pause) table
FUNCTION_WORDS = set("a an the and or but of to in on at by for with as is was were be that this it from".split())
PUNCT_ORDER = [(".", "period"), ("?", "question"), ("!", "exclaim"), (";", "semicolon"),
               (":", "colon"), (",", "comma"), ("-", "dash"), ('"', "quote")]
PUNCT_CATEGORIES = sorted({name for _, name in PUNCT_ORDER} | {"none"})


def trailing_punct(token):
    m = re.search(r"[A-Za-z']([^A-Za-z']*)$", token)
    tail = m.group(1) if m else token
    for ch, name in PUNCT_ORDER:
        if ch in tail:
            return name
    return "none"


def text_words(text):
    """[(clean_word, punctuation_after)] from the transcript; hyphenated words are split."""
    out = []
    for tok in text.replace("--", " -- ").split():
        parts = [p for p in re.split(r"-", tok) if re.search(r"[A-Za-z]", p)]
        if not parts:                                   # standalone "--" etc.
            if out:
                out[-1] = (out[-1][0], "dash")
            continue
        for k, p in enumerate(parts):
            clean = re.sub(r"[^a-z']", "", p.lower())
            out.append((clean, trailing_punct(tok) if k == len(parts) - 1 else "none"))
    return out


def gap_rows(u, text, align_dir, min_pause_sec):
    """One row per word gap. Uses MFA only for the word list and the pause LABEL;
    every feature is computable from text alone."""
    words, _ = read_textgrid(align_dir, u)
    mw, plen = [], []
    for iv in words:
        if iv.mark:
            mw.append(re.sub(r"[^a-z']", "", iv.mark.lower())); plen.append(0.0)
        elif mw:
            plen[-1] = iv.maxTime - iv.minTime
    tw = text_words(text)
    punct = ["none"] * len(mw)
    matcher = difflib.SequenceMatcher(a=mw, b=[w for w, _ in tw], autojunk=False)
    for a, b, size in matcher.get_matching_blocks():
        for k in range(size):
            punct[a + k] = tw[b + k][1]
    rows, since, since_ch, n = [], 0, 0, len(mw)
    for j in range(n - 1):
        since_ch += len(mw[j])
        rows.append(dict(utt=u, gap=j, cur_w=mw[j], next_w=mw[j + 1], punct=punct[j],
                         wlen=len(mw[j]), next_wlen=len(mw[j + 1]), rel_pos=j / n, n_words=n,
                         to_end=n - j - 1, since_punct=since, since_chars=since_ch,
                         plen=plen[j], pause=int(plen[j] >= min_pause_sec)))
        if punct[j] != "none":
            since, since_ch = 0, 0
        else:
            since += 1
    return rows


def build_gap_table(meta, align_dir, min_pause_sec):
    text_of = dict(zip(meta.id, meta.text))
    rows = []
    for u in tqdm(sorted(text_of), desc="gap table"):
        if os.path.exists(os.path.join(align_dir, f"{u}.TextGrid")):
            rows += gap_rows(u, text_of[u], align_dir, min_pause_sec)
    return pd.DataFrame(rows)

In [ ]:
"""Data splits.

* Our split: 90/10 by utterance (GroupShuffleSplit, seed 42).
* SpeechBrain split: the pretrained FastSpeech2 and HiFi-GAN were trained on a 90/10
  train/valid split of LJSpeech (seed 1234). We re-run SpeechBrain's own split function
  to recover which sentences they never trained on.
* Leak-free evaluation set: sentences in OUR test split AND in SpeechBrain's held-out set.
"""
import ast
import csv
import logging
import os
import random
import urllib.request

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


def make_split(utts, test_size, seed):
    utts = np.array(sorted(utts))
    _, te = next(GroupShuffleSplit(test_size=test_size, random_state=seed).split(utts, groups=utts))
    split = pd.DataFrame({"utt": utts, "split": "train"})
    split.loc[te, "split"] = "test"
    return split


def speechbrain_heldout(data_root, prepare_url, seed=1234, ratio=(90, 10), cache="ljspeech_prepare.py"):
    """Run SpeechBrain's split_sets() with their seed; returns the set of held-out utterance ids."""
    if not os.path.exists(cache):
        urllib.request.urlretrieve(prepare_url, cache)
    src = open(cache).read()
    tree = ast.parse(src)
    ns = {"random": random, "os": os, "csv": csv, "logger": logging.getLogger("sb")}
    for node in tree.body:                                   # module constants (e.g. METADATA_CSV)
        if isinstance(node, ast.Assign):
            try:
                exec(compile(ast.Module([node], []), "const", "exec"), ns)
            except Exception:
                pass
    fn = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == "split_sets")
    exec(compile(ast.Module([fn], []), "split_sets", "exec"), ns)
    random.seed(seed)                     # prepare_ljspeech seeds right before calling split_sets
    data_split, meta_csv = ns["split_sets"](data_root, ["train", "valid"], list(ratio))
    return {meta_csv[i][0] for i in data_split["valid"]}


def leak_free_eval_set(split, heldout, n, seed):
    pool = sorted(set(split[split.split == "test"].utt) & set(heldout))
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(pool, n, replace=False)), len(pool)

In [ ]:
import numpy as np, pandas as pd

P, A = cfg["paths"], cfg["audio"]
meta = load_metadata(P["data_root"])

dur = build_duration_table(P["align_dir"], A["sr"], A["hop_length"])
print(f"{dur.utt.nunique()} aligned utterances | {13100 - dur.utt.nunique()} failed alignments")
dur.to_csv(work_path(cfg, "dur_full.csv"), index=False)

split = make_split(dur.utt.unique(), cfg["split"]["test_size"], cfg["split"]["seed"])
split.to_csv(results_path(cfg, "split.csv"), index=False)

sb = cfg["speechbrain_split"]
heldout = speechbrain_heldout(P["data_root"], sb["prepare_script_url"], sb["seed"], sb["ratio"])
pd.DataFrame({"utt": sorted(heldout)}).to_csv(results_path(cfg, "speechbrain_heldout.csv"), index=False)

ev = cfg["eval"]
eval_utts, pool = leak_free_eval_set(split, heldout, ev["n_sentences"], ev["seed"])
pd.DataFrame({"utt": eval_utts}).to_csv(results_path(cfg, "eval_utts_clean.csv"), index=False)
print(f"SpeechBrain held-out {len(heldout)} | our test {(split.split=='test').sum()} | "
      f"both {pool} | eval set {len(eval_utts)}")

gaps = build_gap_table(meta, P["align_dir"], cfg["pause_model"]["min_pause_sec"])
gaps.to_csv(work_path(cfg, "pause_gaps.csv"), index=False)
print(f"{len(gaps)} word gaps, {gaps.pause.mean()*100:.1f}% with a pause")

# sanity check: MFA frame counts should match the cached mel lengths
if os.path.isdir(P["mel_dir"]):
    sample = dur.utt.drop_duplicates().sample(200, random_state=0)
    mfa = dur.groupby("utt").frames.sum()
    diff = [np.load(f"{P['mel_dir']}/{u}.npy", mmap_mode="r").shape[1] - mfa[u] for u in sample]
    print("mel - MFA frames (200 clips): min", min(diff), "max", max(diff))

## 4. AML model 1: duration predictor
LightGBM regressor on phoneme identity, neighbours, stress, position in word/sentence and pause context; compared with a per-phoneme mean.

In [ ]:
"""AML baseline part 1: predict each phoneme's duration (in mel frames) from context."""
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

CAT_COLS = ["phone", "prev", "next"]
NUM_COLS = ["stress", "pos_in_word", "word_len", "before_pause", "rel_pos", "n_phones"]


class DurationModel:
    def __init__(self, n_estimators=600, learning_rate=0.05, num_leaves=63):
        self.params = dict(n_estimators=n_estimators, learning_rate=learning_rate,
                           num_leaves=num_leaves, verbose=-1)
        self.cats, self.model = None, None

    def features(self, df):
        X = df[CAT_COLS + NUM_COLS].copy()
        for c in CAT_COLS:
            X[c] = pd.Categorical(X[c], categories=self.cats[c])
        return X

    def fit(self, train_rows, all_rows):
        """all_rows only fixes the category vocabulary (no labels are used from it)."""
        self.cats = {c: sorted(all_rows[c].unique()) for c in CAT_COLS}
        self.model = lgb.LGBMRegressor(**self.params).fit(self.features(train_rows), train_rows.frames)
        return self

    def predict(self, df):
        return self.model.predict(self.features(df))

    def predict_frames(self, df):
        """Integer frames; at least 1 frame for speech sounds, 0 allowed for pauses."""
        p = np.round(self.predict(df)).astype(int)
        return [max(v, 0 if ph == "sil" else 1) for v, ph in zip(p, df.phone)]

    def save(self, path):
        joblib.dump({"params": self.params, "cats": self.cats, "model": self.model}, path)

    @classmethod
    def load(cls, path):
        obj = joblib.load(path)
        m = cls(); m.params, m.cats, m.model = obj["params"], obj["cats"], obj["model"]
        return m


def phone_mean_baseline(train_rows, test_rows):
    """Simplest baseline: every phoneme gets its average training duration."""
    means = train_rows.groupby("phone").frames.mean()
    return test_rows.phone.map(means).fillna(train_rows.frames.mean()).values

## 5. AML model 2: pause classifier
Predicts, from text only, whether the speaker pauses at each word gap; compared with the rule *pause iff punctuation*.
The decision threshold is tuned on part of the training data.

In [ ]:
"""AML baseline part 2: decide, from text only, whether the speaker pauses at each word gap."""
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


GAP_COLS = ["punct", "cur_w_c", "next_w_c", "wlen", "next_wlen", "rel_pos", "n_words",
            "to_end", "since_punct", "since_chars"]


class PauseModel:
    def __init__(self, top_words=80, n_estimators=500, learning_rate=0.05, num_leaves=63):
        self.top_words = top_words
        self.params = dict(n_estimators=n_estimators, learning_rate=learning_rate,
                           num_leaves=num_leaves, verbose=-1)
        self.top, self.cats, self.model, self.threshold = None, None, None, 0.5

    def _prepare(self, g):
        g = g.copy()
        for c in ("cur_w", "next_w"):
            g[c + "_c"] = g[c].where(g[c].isin(self.top), "<other>")
        X = g[GAP_COLS].copy()
        for c, cats in self.cats.items():
            X[c] = pd.Categorical(X[c], categories=cats)
        return X

    def fit(self, train_gaps, dev_fraction=0.1, seed=0):
        """Train on most of TRAIN; tune the decision threshold for F1 on a held-out part of TRAIN."""
        self.top = (set(train_gaps.cur_w.value_counts().head(self.top_words).index)
                    | set(train_gaps.next_w.value_counts().head(self.top_words).index))
        words = sorted(self.top | {"<other>"})
        self.cats = {"punct": PUNCT_CATEGORIES, "cur_w_c": words, "next_w_c": words}
        utts = train_gaps.utt.unique()
        rng = np.random.default_rng(seed)
        dev_utts = set(rng.choice(utts, int(len(utts) * dev_fraction), replace=False))
        tr, dev = train_gaps[~train_gaps.utt.isin(dev_utts)], train_gaps[train_gaps.utt.isin(dev_utts)]
        self.model = lgb.LGBMClassifier(**self.params).fit(self._prepare(tr), tr.pause)
        proba = self.model.predict_proba(self._prepare(dev))[:, 1]
        self.threshold = max(np.arange(0.2, 0.71, 0.05), key=lambda t: f1_score(dev.pause, proba > t))
        return self

    def predict(self, gaps):
        if len(gaps) == 0:
            return np.zeros(0, dtype=int)
        return (self.model.predict_proba(self._prepare(gaps))[:, 1] > self.threshold).astype(int)

    def save(self, path):
        joblib.dump(self.__dict__, path)

    @classmethod
    def load(cls, path):
        m = cls(); m.__dict__.update(joblib.load(path))
        return m


def punctuation_rule(gaps):
    """Baseline: pause if and only if there is punctuation after the word."""
    return (gaps.punct != "none").astype(int).values

In [ ]:
import numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, mean_absolute_error, precision_recall_fscore_support

ms = frame_ms(cfg)
dur = pd.read_csv(work_path(cfg, "dur_full.csv"), keep_default_na=False)
split = pd.read_csv(results_path(cfg, "split.csv"))
d = dur[dur.edge_sil == 0].merge(split, on="utt")       # leading/trailing silence excluded
train, test = d[d.split == "train"], d[d.split == "test"]

# ---------------- duration predictor
dm = DurationModel(**cfg["duration_model"]).fit(train, d)
dm.save(work_path(cfg, "duration_model.pkl"))
preds = {"phone_mean": phone_mean_baseline(train, test), "lightgbm": dm.predict(test)}
sil = (test.phone == "sil").values
rows = []
for name, p in preds.items():
    rows.append(dict(model=name,
                     mae_all_ms=mean_absolute_error(test.frames, p) * ms,
                     mae_speech_ms=mean_absolute_error(test.frames[~sil], p[~sil]) * ms,
                     mae_pause_ms=mean_absolute_error(test.frames[sil], p[sil]) * ms))
dur_res = pd.DataFrame(rows); dur_res.to_csv(results_path(cfg, "duration_metrics.csv"), index=False)
print(dur_res.round(1).to_string(index=False))
imp = pd.Series(dm.model.booster_.feature_importance("gain"), index=dm.features(test).columns)
imp.sort_values(ascending=False).to_csv(results_path(cfg, "duration_feature_importance.csv"))

# ---------------- pause classifier
gaps = pd.read_csv(work_path(cfg, "pause_gaps.csv"), keep_default_na=False).merge(split, on="utt")
pc = cfg["pause_model"]
pm = PauseModel(pc["top_words"], pc["n_estimators"], pc["learning_rate"], pc["num_leaves"])
pm.fit(gaps[gaps.split == "train"], pc["dev_fraction"], pc["seed"])
pm.save(work_path(cfg, "pause_model.pkl"))
te = gaps[gaps.split == "test"]
rows = []
for name, p in [("punctuation_rule", punctuation_rule(te)), ("lightgbm", pm.predict(te))]:
    pr, rc, f1, _ = precision_recall_fscore_support(te.pause, p, average="binary")
    rows.append(dict(model=name, accuracy=accuracy_score(te.pause, p), precision=pr, recall=rc, f1=f1))
pause_res = pd.DataFrame(rows); pause_res.to_csv(results_path(cfg, "pause_metrics.csv"), index=False)
print(f"pause threshold {pm.threshold:.2f}\n", pause_res.round(3).to_string(index=False))

## 6. Synthesis systems
| System | Pauses | Durations | Sound |
|---|---|---|---|
| `baseline_*` | MFA / MFA / predicted | MFA / LightGBM / LightGBM | averaged mel per phoneme |
| `hybrid_*` | same | same | FastSpeech2 decoder conditioned on our durations |
| `fastspeech2` | predicted | FastSpeech2's own | FastSpeech2 decoder |
| `ceiling` | real | real | real mel → vocoder |

**Input format matters:** the SpeechBrain FastSpeech2 vocabulary has no word-separator token; pauses are `spn`.
A space token silently becomes `<unk>` and badly degrades output, so `Synthesizer.encode` refuses unknown tokens.

In [ ]:
"""Speech synthesis for every system in the comparison.

A *plan* is a phoneme sequence (labels without stress, 'sil' for pauses) plus a duration
in mel frames for each phoneme. Plans come from:
  * real alignment      (MFA phonemes + MFA pauses + MFA or predicted durations)
  * text only           (MFA pronunciations + PREDICTED pauses + PREDICTED durations)
A plan is rendered either by
  * the duration-only baseline: one averaged mel frame per phoneme, repeated (AML only)
  * the FastSpeech2 decoder, conditioned on the plan's durations (hybrid: AML timing + DL sound)

IMPORTANT input format for the SpeechBrain FastSpeech2 checkpoint: its vocabulary has the
39 ARPAbet phonemes, 'spn' (used for pauses) and '<unk>'. There is NO space/word-boundary
token - feeding ' ' silently maps to '<unk>' and badly degrades the output.
"""
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm



# ------------------------------------------------------------------ plans
def text_only_plan(u, text, cfg, dur_model, pause_model):
    """Phonemes from MFA pronunciations; pauses and durations predicted from text only."""
    words, phones = read_textgrid(cfg["paths"]["align_dir"], u)
    words = [w for w in words if w.mark]
    wph = [[p.mark for p in phones if w.minTime <= p.minTime < w.maxTime and p.mark] for w in words]
    gaps = pd.DataFrame(gap_rows(u, text, cfg["paths"]["align_dir"], cfg["pause_model"]["min_pause_sec"]))
    pause = pause_model.predict(gaps)
    seq = []
    for j, ph in enumerate(wph):
        for k, mark in enumerate(ph):
            seq.append((mark, k, len(ph)))
        if j < len(wph) - 1 and pause[j]:
            seq.append(("", -1, 0))
    labels = [("".join(c for c in s[0] if not c.isdigit()) or "sil") for s in seq]
    L, n = len(seq), len(seq) + 2                 # +2: usual leading/trailing silence
    feats = pd.DataFrame([dict(
        phone=labels[i], stress=int(s[0][-1]) if s[0][-1:].isdigit() else -1,
        prev=labels[i - 1] if i > 0 else "sil", next=labels[i + 1] if i < L - 1 else "sil",
        pos_in_word=s[1], word_len=s[2], before_pause=int(i == L - 1 or labels[i + 1] == "sil"),
        rel_pos=(i + 1) / n, n_phones=n) for i, s in enumerate(seq)])
    durs = np.maximum(np.round(dur_model.predict(feats)).astype(int), 1)
    return labels, [int(d) for d in durs], int(pause.sum())


def to_fs2_tokens(labels, durs, strip=True):
    """FastSpeech2 input: phonemes only, consecutive pauses merged into one 'spn' token."""
    toks, td = [], []
    for lab, d in zip(labels, durs):
        if lab in ("sil", "spn"):
            if toks and toks[-1] == "spn":
                td[-1] += int(d)
            else:
                toks.append("spn"); td.append(int(d))
        else:
            toks.append(lab); td.append(int(d))
    if strip:
        while toks and toks[0] == "spn":
            toks.pop(0); td.pop(0)
        while toks and toks[-1] == "spn":
            toks.pop(); td.pop()
    return toks, td


# ------------------------------------------------------------------ averaged mels (baseline)
def compute_avg_mels(cfg, dur_table, train_utts):
    sums, counts = defaultdict(lambda: np.zeros(cfg["audio"]["n_mels"])), defaultdict(int)
    for u, rows in tqdm(dur_table[dur_table.utt.isin(train_utts)].groupby("utt", sort=False),
                        desc="average mels"):
        mel, start = load_mel(cfg, u), 0
        for ph, n in zip(rows.sort_values("idx").phone, rows.sort_values("idx").frames):
            seg = mel[:, start:min(start + n, mel.shape[1])]; start += n
            if seg.shape[1]:
                sums[ph] += seg.sum(1); counts[ph] += seg.shape[1]
    avg = {p: torch.tensor(sums[p] / counts[p], dtype=torch.float32) for p in sums}
    global_avg = sum(avg[p] * counts[p] for p in avg) / sum(counts.values())
    return {"avg": avg, "global_avg": global_avg}


# ------------------------------------------------------------------ synthesizer
class Synthesizer:
    def __init__(self, cfg, device="cuda", finetuned_ckpt=None):
        from speechbrain.inference.TTS import FastSpeech2
        from speechbrain.inference.vocoders import HIFIGAN
        dev = {"device": "cuda:0" if device == "cuda" else device}
        self.device = device
        self.hifi = HIFIGAN.from_hparams(source=cfg["models"]["hifigan"], savedir="pretrained/hifigan", run_opts=dev)
        self.fs2 = FastSpeech2.from_hparams(source=cfg["models"]["fastspeech2"], savedir="pretrained/fs2", run_opts=dev)
        self.model = self.fs2.hparams.model
        self.unk = self.fs2.input_encoder.lab2ind["<unk>"]
        if finetuned_ckpt:
            self.load_decoder(finetuned_ckpt)
        self.model.eval()

    def load_decoder(self, ckpt):
        self.model.load_state_dict(torch.load(ckpt, map_location=self.device))
        self.model.eval()

    def encode(self, toks):
        ids = self.fs2.input_encoder.encode_sequence_torch(toks).long()
        if (ids == self.unk).any():
            bad = [t for t, i in zip(toks, ids.tolist()) if i == self.unk]
            raise ValueError(f"tokens not in FastSpeech2 vocabulary: {sorted(set(bad))}")
        return ids

    @torch.no_grad()
    def vocode(self, mel):                         # mel: (n_mels, T) or (1, n_mels, T)
        mel = mel if mel.dim() == 3 else mel.unsqueeze(0)
        return norm_audio(self.hifi.decode_batch(mel.to(self.device)).squeeze().cpu().numpy())

    @torch.no_grad()
    def fastspeech2(self, toks, durs=None):
        """Render tokens with the FastSpeech2 decoder. durs=None -> its own duration predictor.
        Returns (waveform, own predicted durations in frames, number of mel frames)."""
        ids = self.encode(toks).unsqueeze(0).to(self.device)
        dur = None if durs is None else torch.tensor(durs, dtype=torch.long).unsqueeze(0).to(self.device)
        out = self.model(ids, durations=dur)
        mel = out[1].transpose(-1, 1)                                  # postnet output (1, n_mels, T)
        own = np.clip(np.exp(out[2].squeeze(0).cpu().numpy()) - 1, 0, None)   # log-duration -> frames
        return self.vocode(mel), own, mel.shape[-1]

    def baseline(self, labels, durs, avg_mels):
        avg, g = avg_mels["avg"], avg_mels["global_avg"]
        frames = [avg.get(p, g).unsqueeze(1).repeat(1, max(int(d), 0)) for p, d in zip(labels, durs)]
        return self.vocode(torch.cat(frames, dim=1))

    def ceiling(self, mel_np):
        """Real mel -> vocoder: the best this vocoder can do."""
        return self.vocode(torch.from_numpy(mel_np))

In [ ]:
dur = pd.read_csv(work_path(cfg, "dur_full.csv"), keep_default_na=False)
split = pd.read_csv(results_path(cfg, "split.csv"))
# ---------------- averaged mel per phoneme (duration-only baseline)
torch.save(compute_avg_mels(cfg, dur, set(split[split.split == "train"].utt)), work_path(cfg, "avg_mels.pt"))
print("saved models and averaged mels")

## 7. Fine-tuning the decoder
Decoder + output layer + postnet are trained on **our** training split with **our** MFA durations (encoder and variance predictors frozen); best epoch chosen on 200 held-out training sentences.

In [ ]:
"""Fine-tune the FastSpeech2 decoder + output layer + postnet on OUR training split,
conditioned on OUR MFA durations. Encoder and variance predictors stay frozen."""
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm



def make_item(cfg, synth, rows):
    """(token ids, durations, target mel (T, n_mels)) for one utterance, edge silences cropped."""
    rows = rows.sort_values("idx")
    u = rows.utt.iloc[0]
    lead = int(rows.frames[(rows.edge_sil == 1) & (rows.idx == 0)].sum())
    trail = int(rows.frames[(rows.edge_sil == 1) & (rows.idx == rows.n_phones - 1)].sum())
    toks, td = [], []
    for ph, fr in zip(rows.phone[rows.edge_sil == 0], rows.frames[rows.edge_sil == 0]):
        if ph in ("sil", "spn"):
            if toks and toks[-1] == "spn":
                td[-1] += int(fr)
            else:
                toks.append("spn"); td.append(int(fr))
        else:
            toks.append(ph); td.append(int(fr))
    mel = load_mel(cfg, u)
    mel = mel[:, lead: mel.shape[1] - trail]
    if not toks or sum(td) != mel.shape[1]:
        return None
    return (synth.encode(toks), torch.tensor(td, dtype=torch.long),
            torch.from_numpy(mel.T.copy()).half())


def batches(items, bs, device, shuffle=True):
    order = sorted(range(len(items)), key=lambda i: items[i][2].shape[0])
    chunks = [order[i:i + bs] for i in range(0, len(order), bs)]
    if shuffle:
        random.shuffle(chunks)
    for c in chunks:
        t, d, m = zip(*[items[i] for i in c])
        T = max(x.shape[0] for x in m)
        yield (torch.nn.utils.rnn.pad_sequence(t, batch_first=True, padding_value=0).to(device),
               torch.nn.utils.rnn.pad_sequence(d, batch_first=True, padding_value=0).to(device),
               torch.stack([F.pad(x.float(), (0, 0, 0, T - x.shape[0])) for x in m]).to(device),
               torch.tensor([x.shape[0] for x in m], device=device))


def masked_l1(pred, tgt, lens):
    T = min(pred.shape[1], tgt.shape[1])
    mask = (torch.arange(T, device=lens.device)[None] < lens[:, None]).unsqueeze(-1)
    return ((pred[:, :T] - tgt[:, :T]).abs() * mask).sum() / (mask.sum() * tgt.shape[-1])


def finetune_decoder(cfg, synth, dur_table, train_utts, ckpt_path, device="cuda"):
    f = cfg["finetune"]
    random.seed(f["seed"]); torch.manual_seed(f["seed"])
    utts = sorted(train_utts); random.Random(f["seed"]).shuffle(utts)
    dev_utts, trn_utts = utts[: f["dev_size"]], utts[f["dev_size"]:]
    groups = {u: g for u, g in dur_table[dur_table.utt.isin(train_utts)].groupby("utt", sort=False)}
    trn = [x for x in (make_item(cfg, synth, groups[u]) for u in tqdm(trn_utts, desc="train items")) if x]
    dev = [x for x in (make_item(cfg, synth, groups[u]) for u in tqdm(dev_utts, desc="dev items")) if x]
    print(f"{len(trn)} train / {len(dev)} dev items")

    model = synth.model
    for p in model.parameters():
        p.requires_grad = False
    train_mods = [model.decoder, model.linear, model.postnet]
    for mod in train_mods:
        for p in mod.parameters():
            p.requires_grad = True
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"trainable: {sum(p.numel() for p in params)/1e6:.1f}M / {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

    @torch.no_grad()
    def dev_loss():
        model.eval(); tot, n = 0.0, 0
        for tok, dur, mel, lens in batches(dev, f["batch_size"], device, shuffle=False):
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(tok, durations=dur)
            tot += masked_l1(out[1].float(), mel, lens).item() * len(lens); n += len(lens)
        return tot / n

    opt = torch.optim.AdamW(params, lr=f["lr"], weight_decay=f["weight_decay"])
    scaler = torch.amp.GradScaler("cuda")
    best = dev_loss()
    log = [dict(epoch=0, train_loss=np.nan, dev_l1=best)]
    print(f"pretrained dev L1: {best:.4f}")
    torch.save(model.state_dict(), ckpt_path)

    for ep in range(1, f["epochs"] + 1):
        model.eval()
        for mod in train_mods:
            mod.train()
        t0, run = time.time(), []
        for tok, dur, mel, lens in tqdm(list(batches(trn, f["batch_size"], device)), desc=f"epoch {ep}"):
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(tok, durations=dur)
                loss = masked_l1(out[0].float(), mel, lens) + masked_l1(out[1].float(), mel, lens)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, f["grad_clip"])
            scaler.step(opt); scaler.update(); run.append(loss.item())
        dl = dev_loss()
        log.append(dict(epoch=ep, train_loss=float(np.mean(run)), dev_l1=dl))
        print(f"epoch {ep}: train {np.mean(run):.4f} | dev L1 {dl:.4f} | {time.time()-t0:.0f}s")
        if dl < best:
            best = dl; torch.save(model.state_dict(), ckpt_path); print("  saved best")

    synth.load_decoder(ckpt_path)
    return pd.DataFrame(log)

In [ ]:
import pandas as pd

dur = pd.read_csv(work_path(cfg, "dur_full.csv"), keep_default_na=False)
split = pd.read_csv(results_path(cfg, "split.csv"))
synth = Synthesizer(cfg, device())
log = finetune_decoder(cfg, synth, dur, set(split[split.split == "train"].utt),
                       work_path(cfg, "fs2_finetuned.pt"), device())
log.to_csv(results_path(cfg, "finetune_log.csv"), index=False)
print(log.round(4).to_string(index=False))

## 8. Rendering every system on the 100 leak-free sentences

In [ ]:
import numpy as np, pandas as pd, soundfile as sf, torch
from tqdm.auto import tqdm

P, sr = cfg["paths"], cfg["audio"]["sr"]
text_of = dict(zip(*load_metadata(P["data_root"])[["id", "text"]].values.T))
dur = pd.read_csv(work_path(cfg, "dur_full.csv"), keep_default_na=False)
eval_utts = pd.read_csv(results_path(cfg, "eval_utts_clean.csv")).utt.tolist()
dm = DurationModel.load(work_path(cfg, "duration_model.pkl"))
pm = PauseModel.load(work_path(cfg, "pause_model.pkl"))
avg = torch.load(work_path(cfg, "avg_mels.pt"))
synth = Synthesizer(cfg, device())
save = lambda s, u, w: sf.write(work_path(cfg, "audio", s, f"{u}.wav"), w, sr)

plans = {}
for u in eval_utts:
    rows = dur[(dur.utt == u) & (dur.edge_sil == 0)].sort_values("idx")
    labels = list(rows.phone)
    tl, td, _ = text_only_plan(u, text_of[u], cfg, dm, pm)
    plans[u] = dict(rows=rows, true=(labels, list(rows.frames)),
                    pred=(labels, dm.predict_frames(rows)), text=(tl, td))

def render_neural(suffix):
    comp = []
    for u in tqdm(eval_utts, desc="neural" + suffix):
        p = plans[u]
        for kind in ("true", "pred", "text"):
            toks, d = to_fs2_tokens(*p[kind])
            save(f"hybrid_{kind}{suffix}", u, synth.fastspeech2(toks, d)[0])
        toks, _ = to_fs2_tokens(*p["text"])
        save(f"fastspeech2{suffix}", u, synth.fastspeech2(toks)[0])     # its own durations
        # duration accuracy on speech sounds, identical tokens for both predictors
        toks, d = to_fs2_tokens(*p["true"])
        own = synth.fastspeech2(toks)[1]
        keep = [i for i, t in enumerate(toks) if t != "spn"]
        sp = p["rows"][~p["rows"].phone.isin(["sil", "spn"])]
        comp.append(pd.DataFrame(dict(utt=u, phone=sp.phone.values, true=sp.frames.values,
                                      lightgbm=dm.predict(sp), fastspeech2=own[keep])))
    return pd.concat(comp)

# ---------------- AML-only baselines and the vocoder ceiling
for u in tqdm(eval_utts, desc="baselines"):
    save("ceiling", u, synth.ceiling(load_mel(cfg, u)))
    for kind in ("true", "pred", "text"):
        save(f"baseline_{kind}", u, synth.baseline(*plans[u][kind], avg))

# ---------------- neural decoder: pretrained, then fine-tuned
comp = render_neural("")
comp.to_csv(results_path(cfg, "duration_comparison.csv"), index=False)
ckpt = work_path(cfg, "fs2_finetuned.pt")
if os.path.exists(ckpt):
    synth.load_decoder(ckpt)
    render_neural("_ft")
else:
    print("no fine-tuned checkpoint found - skipping *_ft systems")

# ---------------- a few samples for the repo
systems = ["ceiling", "hybrid_text_ft", "hybrid_text", "fastspeech2", "baseline_text"]
for u in eval_utts[: cfg["eval"]["n_samples"]]:
    os.makedirs(os.path.join(P["samples_dir"], u), exist_ok=True)
    shutil.copy(os.path.join(P["data_root"], "wavs", f"{u}.wav"), os.path.join(P["samples_dir"], u, "original.wav"))
    for s in systems:
        src = work_path(cfg, "audio", s, f"{u}.wav")
        if os.path.exists(src):
            shutil.copy(src, os.path.join(P["samples_dir"], u, f"{s}.wav"))
    with open(os.path.join(P["samples_dir"], u, "text.txt"), "w") as f:
        f.write(text_of[u])
print("done")

## 9. Evaluation
WER from a held-out ASR (Whisper `small.en`) with 95% bootstrap CIs over sentences, paired comparisons on the same sentences, duration MAE with identical tokens, and UTMOS (automatic naturalness estimate).

In [ ]:
"""Metrics: WER from a held-out ASR (Whisper), bootstrap confidence intervals,
paired comparisons, UTMOS naturalness, and the human listening-test analysis."""
import numpy as np
import pandas as pd


class Transcriber:
    def __init__(self, model_name="small.en", device="cuda"):
        import whisper
        from whisper.normalizers import EnglishTextNormalizer
        self.asr = whisper.load_model(model_name, device=device)
        self.norm = EnglishTextNormalizer()
        self.fp16 = device == "cuda"

    def __call__(self, path):
        return self.norm(self.asr.transcribe(path, language="en", fp16=self.fp16)["text"])


def error_counts(refs, hyps):
    """Per-sentence (errors, reference words) - needed for sentence-level bootstrap."""
    import jiwer
    e, n = [], []
    for r, h in zip(refs, hyps):
        o = jiwer.process_words(r, h if h.strip() else "<empty>")
        e.append(o.substitutions + o.deletions + o.insertions); n.append(len(r.split()))
    return np.array(e), np.array(n)


def bootstrap_wer(counts, n_boot=5000, seed=0):
    """counts: {system: (errors, words)}. Same resampled sentences for every system (paired)."""
    n = len(next(iter(counts.values()))[0])
    idx = np.random.default_rng(seed).integers(0, n, (n_boot, n))
    return {s: e[idx].sum(1) / w[idx].sum(1) for s, (e, w) in counts.items()}


def ci(samples, q=(2.5, 97.5)):
    lo, hi = np.percentile(samples, q)
    return float(lo), float(hi)


class UTMOS:
    """Automatic MOS predictor (1-5). An estimate of naturalness, not a substitute for listeners."""

    def __init__(self, repo, model, device="cuda"):
        import torch
        self.device = device
        self.model = torch.hub.load(repo, model, trust_repo=True).to(device).eval()

    def __call__(self, path):
        import soundfile as sf
        import torch
        wav, sr = sf.read(path, dtype="float32")
        with torch.no_grad():
            return self.model(torch.from_numpy(wav).unsqueeze(0).to(self.device), sr).item()


def bootstrap_mean(values, n_boot=5000, seed=0):
    values = np.asarray(values)
    idx = np.random.default_rng(seed).integers(0, len(values), (n_boot, len(values)))
    return values[idx].mean(1)


# ------------------------------------------------------------------ listening test
def analyse_listening_test(responses, key, n_sentences=10, n_boot=5000, seed=0):
    """responses: Google Forms CSV (personal columns already dropped).
    key: sentence, label (A/B), system. Returns per-judgement table and a summary."""
    rows = []
    for s in range(n_sentences):
        suf = "" if s == 0 else f".{s}"
        a_sys = key[(key.sentence == s + 1) & (key.label == "A")].system.iloc[0]
        for r, x in responses.iterrows():
            choice = x["Which voice was easier to understand?" + suf]
            score = {"A": x["How easy was Voice A to understand?" + suf],
                     "B": x["How easy was Voice B to understand?" + suf]}
            sys_of = {"A": a_sys, "B": key[(key.sentence == s + 1) & (key.label == "B")].system.iloc[0]}
            pref = "tie" if choice == "No difference" else sys_of[choice[-1]]
            rows.append(dict(sentence=s + 1, rater=r, preferred=pref,
                             **{f"score_{sys_of[k]}": v for k, v in score.items()}))
    d = pd.DataFrame(rows)
    systems = [c[len("score_"):] for c in d.columns if c.startswith("score_")]
    rng, raters = np.random.default_rng(seed), d.rater.unique()
    boot = []
    for _ in range(n_boot):          # resample RATERS (judgements of one rater are not independent)
        dd = pd.concat([d[d.rater == q] for q in rng.choice(raters, len(raters))])
        boot.append((dd[f"score_{systems[0]}"] - dd[f"score_{systems[1]}"]).mean())
    summary = dict(n_raters=len(raters), n_judgements=len(d),
                   **{f"preferred_{s}": int((d.preferred == s).sum()) for s in systems},
                   ties=int((d.preferred == "tie").sum()),
                   **{f"mean_score_{s}": float(d[f"score_{s}"].mean()) for s in systems},
                   score_diff=f"{systems[0]} - {systems[1]}",
                   score_diff_mean=float((d[f"score_{systems[0]}"] - d[f"score_{systems[1]}"]).mean()),
                   score_diff_ci=ci(boot))
    return d, summary

In [ ]:
SKIP_UTMOS = False
import numpy as np, pandas as pd
from tqdm.auto import tqdm

P, E, M = cfg["paths"], cfg["eval"], cfg["models"]
text_of = dict(zip(*load_metadata(P["data_root"])[["id", "text"]].values.T))
eval_utts = pd.read_csv(results_path(cfg, "eval_utts_clean.csv")).utt.tolist()

systems = ["original", "ceiling", "hybrid_true", "hybrid_pred", "hybrid_text", "fastspeech2",
           "hybrid_true_ft", "hybrid_pred_ft", "hybrid_text_ft", "fastspeech2_ft",
           "baseline_true", "baseline_pred", "baseline_text"]
path = lambda s, u: (os.path.join(P["data_root"], "wavs", f"{u}.wav") if s == "original"
                     else work_path(cfg, "audio", s, f"{u}.wav"))
systems = [s for s in systems if os.path.exists(path(s, eval_utts[0]))]

# ---------------- WER
asr = Transcriber(M["whisper"], device())
refs = [asr.norm(text_of[u]) for u in eval_utts]
hyps = {s: [asr(path(s, u)) for u in tqdm(eval_utts, desc=s)] for s in systems}
pd.DataFrame({"utt": eval_utts, "reference": refs, **hyps}).to_csv(results_path(cfg, "transcripts.csv"), index=False)
counts = {s: error_counts(refs, hyps[s]) for s in systems}
boot = bootstrap_wer(counts, E["n_bootstrap"], E["seed"])
wer = pd.DataFrame([dict(system=s, wer=c[0].sum() / c[1].sum(), ci_low=ci(boot[s])[0], ci_high=ci(boot[s])[1])
                    for s, c in counts.items()])
wer.to_csv(results_path(cfg, "wer.csv"), index=False)
print((wer.set_index("system") * 100).round(1).to_string())

pairs = [("hybrid_text", "fastspeech2"), ("hybrid_text", "baseline_text"), ("hybrid_true", "hybrid_text"),
         ("hybrid_text_ft", "fastspeech2_ft"), ("hybrid_true_ft", "hybrid_true"),
         ("hybrid_text_ft", "hybrid_text"), ("ceiling", "original")]
paired = pd.DataFrame([dict(a=a, b=b, diff=(boot[a] - boot[b]).mean(),
                            ci_low=ci(boot[a] - boot[b])[0], ci_high=ci(boot[a] - boot[b])[1])
                       for a, b in pairs if a in boot and b in boot])
paired["significant"] = (paired.ci_low > 0) | (paired.ci_high < 0)
paired.to_csv(results_path(cfg, "wer_paired.csv"), index=False)
print(paired.round(4).to_string(index=False))

# ---------------- duration accuracy (identical tokens for both predictors)
comp = pd.read_csv(results_path(cfg, "duration_comparison.csv"))
ms = frame_ms(cfg)
print(f"duration MAE  LightGBM {np.abs(comp.lightgbm - comp.true).mean()*ms:.1f} ms | "
      f"FastSpeech2 {np.abs(comp.fastspeech2 - comp.true).mean()*ms:.1f} ms")

# ---------------- UTMOS (automatic naturalness estimate)
if not SKIP_UTMOS:
    utmos = UTMOS(M["utmos_repo"], M["utmos_model"], device())
    mos_sys = [s for s in ["original", "ceiling", "hybrid_text_ft", "hybrid_true_ft", "fastspeech2_ft",
                           "hybrid_text", "fastspeech2", "baseline_text"] if s in systems]
    mos = pd.DataFrame({s: [utmos(path(s, u)) for u in tqdm(eval_utts, desc=f"utmos {s}")] for s in mos_sys},
                       index=eval_utts)
    mos.to_csv(results_path(cfg, "utmos_per_sentence.csv"))
    summ = pd.DataFrame([dict(system=s, mos=mos[s].mean(), ci_low=ci(bootstrap_mean(mos[s]))[0],
                              ci_high=ci(bootstrap_mean(mos[s]))[1]) for s in mos_sys])
    summ.to_csv(results_path(cfg, "utmos.csv"), index=False)
    print(summ.round(2).to_string(index=False))

## 10. Human listening test
Blind A/B test run with Google Forms (7 listeners × 10 sentences, random A/B order). The anonymised responses
(no emails, no timestamps) and the answer key are in this repository's `results/` folder.
*Note: this test used the neural system before the input-format fix, so it is a conservative comparison.*

In [ ]:
resp = pd.read_csv(f"{RAW}/results/listening_test_responses_anonymised.csv")
key  = pd.read_csv(f"{RAW}/results/listening_test_key.csv")
judgements, summary = analyse_listening_test(resp, key)
print(json.dumps(summary, indent=2))

## 11. Figures (saved to `outputs/results/figures` for the report and slides)

In [ ]:
figdir = os.path.join(cfg["paths"]["results_dir"], "figures"); os.makedirs(figdir, exist_ok=True)
R = lambda n: pd.read_csv(results_path(cfg, n))
names = {"original": "Original", "ceiling": "Ceiling (real mel)", "hybrid_text_ft": "Hybrid (ours, fine-tuned)",
         "fastspeech2_ft": "FastSpeech2 (own timing)", "baseline_text": "Duration-only baseline"}

# 1) WER with confidence intervals
w = R("wer.csv").set_index("system").loc[list(names)]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh([names[s] for s in w.index], w.wer * 100,
        xerr=[(w.wer - w.ci_low) * 100, (w.ci_high - w.wer) * 100], color="#2b5797", capsize=3)
ax.invert_yaxis(); ax.set_xlabel("WER (%) - lower is better"); ax.set_title("Intelligibility (Whisper WER, 95% CI)")
plt.tight_layout(); plt.savefig(f"{figdir}/wer.png", dpi=200); plt.show()

# 2) Fine-tuning curve
ft = R("finetune_log.csv")
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(ft.epoch, ft.dev_l1, "o-", color="#2b5797"); ax.set_xlabel("epoch (0 = pretrained)"); ax.set_ylabel("dev mel L1")
ax.set_title("Decoder fine-tuning"); plt.tight_layout(); plt.savefig(f"{figdir}/finetune.png", dpi=200); plt.show()

# 3) Duration MAE
dm_ = R("duration_metrics.csv"); comp = R("duration_comparison.csv"); ms = frame_ms(cfg)
vals = {"Phone mean": dm_.set_index("model").loc["phone_mean", "mae_speech_ms"],
        "LightGBM": dm_.set_index("model").loc["lightgbm", "mae_speech_ms"],
        "FastSpeech2 (own)": np.abs(comp.fastspeech2 - comp.true).mean() * ms}
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(list(vals), list(vals.values()), color=["#999", "#2b5797", "#c0504d"])
ax.set_ylabel("MAE (ms), speech sounds"); ax.set_title("Duration prediction error")
plt.tight_layout(); plt.savefig(f"{figdir}/duration_mae.png", dpi=200); plt.show()

# 4) Intelligibility vs naturalness
if os.path.exists(results_path(cfg, "utmos.csv")):
    u = R("utmos.csv").set_index("system"); w_all = R("wer.csv").set_index("system")
    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    for s in u.index:
        ax.scatter(w_all.loc[s, "wer"] * 100, u.loc[s, "mos"], s=60)
        ax.annotate(names.get(s, s), (w_all.loc[s, "wer"] * 100, u.loc[s, "mos"]), fontsize=8,
                    xytext=(4, 4), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("WER (%) - log scale"); ax.set_ylabel("UTMOS (1-5)")
    ax.set_title("Intelligibility vs naturalness"); plt.tight_layout(); plt.savefig(f"{figdir}/wer_vs_utmos.png", dpi=200); plt.show()

# 5) Spectrograms of the same sentence
u0 = R("eval_utts_clean.csv").utt[0]; ext = MelExtractor(cfg["audio"], DEVICE)
paths = {"Original": os.path.join(cfg["paths"]["data_root"], "wavs", f"{u0}.wav"),
         "Hybrid (ours)": work_path(cfg, "audio", "hybrid_text_ft", f"{u0}.wav"),
         "Duration-only baseline": work_path(cfg, "audio", "baseline_text", f"{u0}.wav")}
fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for ax, (t, p) in zip(axes, paths.items()):
    ax.imshow(ext(p).cpu().numpy(), origin="lower", aspect="auto", cmap="magma"); ax.set_title(t, fontsize=10)
    ax.set_ylabel("mel bin")
axes[-1].set_xlabel("frame"); plt.tight_layout(); plt.savefig(f"{figdir}/spectrograms.png", dpi=200); plt.show()

## 12. Listen to samples

In [ ]:
for d in sorted(glob.glob(os.path.join(cfg["paths"]["samples_dir"], "LJ*"))):
    print("\n" + open(f"{d}/text.txt").read())
    for f in sorted(glob.glob(f"{d}/*.wav")):
        print("  ", os.path.basename(f)[:-4]); display(Audio(f))

In [ ]:
# bundle small outputs (results, figures, samples) + fine-tuned checkpoint to download
!cd /content && zip -qr outputs.zip outputs && cp {cfg["paths"]["work_dir"]}/fs2_finetuned.pt /content/ 2>/dev/null; ls -lh /content/outputs.zip

---
## Appendix A: running forced alignment yourself
Use a **separate, fresh Colab runtime** (conda replaces Colab's Python and breaks PyTorch in the same runtime).
Afterwards download `/content/LJ_aligned.tar` and use it in Section 2 instead of the Release file.

In [ ]:
!pip install -q condacolab
import condacolab; condacolab.install()      # restarts the runtime automatically

In [ ]:
!wget -q -nc https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2 -O /content/LJSpeech-1.1.tar.bz2
!tar -xjf /content/LJSpeech-1.1.tar.bz2 -C /content/
!conda install -c conda-forge montreal-forced-aligner -y -q
import csv
with open("/content/LJSpeech-1.1/metadata.csv", encoding="utf-8") as f:
    for row in csv.reader(f, delimiter="|", quoting=csv.QUOTE_NONE):
        open(f"/content/LJSpeech-1.1/wavs/{row[0]}.lab", "w", encoding="utf-8").write(row[2])
!mfa model download acoustic english_us_arpa
!mfa model download dictionary english_us_arpa
!mfa align /content/LJSpeech-1.1/wavs english_us_arpa english_us_arpa /content/LJ_aligned -j 2 --clean
!cd /content && tar -cf LJ_aligned.tar LJ_aligned